# <center><b>Reinforcement Learning for Quadruped Locomotion (Go2)</b></center>

<div align="center">

<h3>KAIST DRCD Lab</h3>


<b>Instructor</b><br>
<a href="https://dynamicrobot.kaist.ac.kr/people.html">Hae-Won Park</a>
(haewonpark@kaist.ac.kr)

<br>

<b>Teaching Assistants</b><br>
<a href="https://kdyun0118.github.io/">Dongyun Kang</a>
(kdong7309@kaist.ac.kr)<br>
<a href="https://github.com/parkjahun42">Jaehyun Park</a>
(parkjahun42@kaist.ac.kr)<br>
<a href="https://github.com/sowoolee">Sowoo Lee</a>
(dlthdn@kaist.ac.kr)


</div>

<br>
<br>

## Abstract

이 튜토리얼은 **MuJoCo 시뮬레이터 환경에서 강화학습을 통해 사족로봇(Unitree Go2)의 보행제어기를 학습**해보는 것을 목표로 합니다.
이를 위해서 PPO(Proximal Policy Optimization)를 기반으로 한 보행 정책 학습 과정을 단계별로 설명하며,

- Go2 MuJoCo 환경 구성 및 관측/행동 설계
- PPO 학습 파이프라인과 주요 하이퍼파라미터
- Policy Training / Evaluation
- Gait phase tracking을 통한 보행 패턴 제어

등의 내용을 포함합니다.

<br>


References

<small>
[1] https://github.com/nimazareian/quadruped-rl-locomotion <br>
[2] Todorov, Emanuel, Tom Erez, and Yuval Tassa. "Mujoco: A physics engine for model-based control." 2012 IEEE/RSJ.
</small>

---

## 0. Environment Setup

본 튜토리얼은 **Google Colab 및 로컬 Jupyter notebook** 환경에서 실행할 수 있도록 작성되었습니다.

아래 순서에 따라 실행 환경을 준비해 주세요.

1. **런타임 유형을 T4 GPU로 설정**하고 런타임에 연결합니다.

2. 제공된 GitHub 레포지토리( https://github.com/agbread/RL_tutorial.git )를 **clone**하여 코드 베이스를 준비합니다.

> Colab이면 `/content`를 기준 경로로 사용하고, 그렇지 않으면 현재 작업 디렉터리를 기준 경로로 사용합니다.
> 그 기준 경로에서 `RL_tutorial`를 클론/사용하도록 동작합니다.

3. MuJoCo 및 강화학습 실험에 필요한 **의존성 패키지들을 설치**합니다.

In [ ]:
# Clone repository
import os, sys

import yaml

# Detect Colab by availability of /content or google.colab.
try:
    import google.colab  # noqa: F401
    in_colab = True
except Exception:
    in_colab = os.path.isdir("/content")

try:
    base_dir
except NameError:
    base_dir = "/content" if in_colab else os.getcwd()
os.chdir(base_dir)

repo_dir = os.path.join(base_dir, "RL_tutorial")

print(f"Base directory: {base_dir}")
print(f"Repo directory: {repo_dir}")

if not os.path.isdir(repo_dir):
  !git clone https://github.com/agbread/RL_tutorial.git
else:
  print("Cloned Directory already exists")

os.chdir(repo_dir)
print("Current Directory: ", os.getcwd())

sys.path.insert(0, os.path.join(repo_dir, "src"))
os.environ["MUJOCO_GL"] = "egl"

# numpy 1.x / 2.x 호환성 패치:
# 저장된 모델이 numpy 2.x (numpy.core_ 경로) 로 직렬화된 경우를 위해
# numpy.core_ 를 numpy.core 의 alias 로 등록합니다.
import numpy, numpy.core, numpy.core.numeric, numpy.core.multiarray
import numpy.random._pickle as _np_pickle

sys.modules['numpy.core_'] = numpy.core
sys.modules['numpy.core_.numeric'] = numpy.core.numeric
sys.modules['numpy.core_.multiarray'] = numpy.core.multiarray

_orig_bg_ctor = _np_pickle.__bit_generator_ctor
def _patched_bg_ctor(bg='MT19937'):
    return bg() if isinstance(bg, type) else _orig_bg_ctor(bg)
_np_pickle.__bit_generator_ctor = _patched_bg_ctor

In [ ]:
# Install dependencies
# stable-baselines3는 PyPI에서 설치합니다 (이 repo에는 sb3 소스 포크가 없음).
# 로컬에서 검증된 버전 조합으로 고정합니다.
!pip install "stable-baselines3==2.3.0" "gymnasium==0.29.1" "mujoco==3.8.0" "numpy<2" "imageio[ffmpeg]" tensorboard pygments

In [ ]:
from pathlib import Path
from IPython.display import HTML
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
import inspect

def _render_code(code, title="code", max_height=400, bg="transparent", indent=16):
    style_name = "native" if in_colab else "friendly"
    formatter = HtmlFormatter(style=style_name, noclasses=True, linenos="inline")
    html = highlight(code, PythonLexer(), formatter)
    css = """
    <style>
    .highlight pre { margin: 0; text-align: left; }
    </style>
    """
    return HTML(f"""
    {css}
    <details>
      <summary>{title}</summary>
      <div style="margin-top:8px; margin-left:{indent}px; max-height:{max_height}px; overflow:auto; border:1px solid #ddd; padding:10px; background:{bg};">
        {html}
      </div>
    </details>
    """)


def show_code(path, max_height=400, bg="transparent"):
    code = Path(path).read_text()
    return _render_code(code, max_height=max_height, bg=bg)

def show_func(obj, max_height=400, bg="transparent"):
    code = inspect.getsource(obj)
    return _render_code(code, max_height=max_height, bg=bg)

---

<br>

## 1. Go2 MuJoCo Environment

본 섹션에서는 **MuJoCo 기반의 Go2 로봇 보행환경**에 대하여 설명합니다.

학습 환경은 `Go2MujocoEnv` 클래스에 구현되어 있으며, 로봇이 정해진 범위 내의 직진 command를 안정적으로 추종하는 문제를 **마르코프 결정 과정(Markov Decision Process, MDP)** 의 형태로 구성합니다.

<br>

### 1.1 Simulation Setup

- **물리 시뮬레이터**: MuJoCo
- **로봇 모델**: Unitree Go2
- **시뮬레이션 정의**:
  시뮬레이션 환경은 `unitree_go2/scene_position.xml` 파일에 정의되어 있으며,
  로봇 모델, 지면, 조명, 카메라 설정을 포함합니다.
- **로봇 설정**:
  관절 액추에이터, PD 제어 이득, 기구학 구조는 `go2.xml` 파일에 정의되어 있습니다.

<br>

In [ ]:
# investigate scene xml files
show_code(f"{repo_dir}/unitree_go2/scene_position.xml", max_height=800)

In [ ]:
# investigate robot xml files
show_code(f"{repo_dir}/unitree_go2/go2.xml", max_height=800)


### 1.2 Environment Configuration

강화학습 환경은 다음과 같이 구현되어 있습니다.

- `Go2MujocoEnv` 클래스 (`src/go2_mujoco_env.py` 참고)
  - 환경의 동역학 및 MDP 구성 로직
  - gymnasium의 MujocoEnv 클래스 상속
  - **Gait phase tracking**: trot gait 패턴을 위한 발 위상(phase) 추적 기능 추가

In [ ]:
show_code(f"{repo_dir}/src/go2_mujoco_env.py", max_height=800)

- `src/envs.yaml`은 환경 설정 파일로, 다음 항목들을 포함합니다.
  - 에피소드 길이
  - Reward 가중치
  - 목표 속도(command) 범위
  - Observation 스케일링
  - 조기 종료(early termination) 조건
  - **Gait 설정** (gait_hz, gait_shift)
  등등

In [ ]:
show_code(f"{repo_dir}/src/envs.yaml", max_height=800)

### 1.3 Environment Step Loop
(`src/go2_mujoco_env.py` 의  `step` method 참고)

<br>

각 타임스텝마다 다음과 같은 절차가 수행됩니다.

1. 정책(policy)이 현재 상태를 기반으로 행동(action)을 출력합니다.
2. 시뮬레이터가 고정된 프레임 수(`frame_skip`)만큼 진행됩니다.
3. **Gait phase**가 업데이트됩니다.
4. Observation, reward, 종료 조건이 계산됩니다.
5. 해당 전이(transition)가 학습을 위해 저장됩니다.


In [ ]:
import importlib
import src.go2_mujoco_env as go2_env

show_func(go2_env.Go2MujocoEnv.step, max_height=800)

### 1.4 Observation Space
(`src/go2_mujoco_env.py` 의  `_get_obs` method 참고)

<br>

Observation 벡터는 다음과 같은 정보를 포함합니다.

- 로봇 베이스의 선형 및 각속도
- 중력 방향이 투영된 벡터
- 목표 이동 속도(command)
- 관절 위치 및 속도
- 이전 타임스텝의 action (제어 입력의 연속성 확보 목적)
- **Gait phase** (sin, cos 인코딩 — trot gait 동기화 목적)

모든 observation은 학습 안정성을 위해 사전에 정의된 범위로 스케일링 및 클리핑됩니다.


In [ ]:
show_func(go2_env.Go2MujocoEnv._get_obs, max_height=800)

### 1.5 Reward and Termination
(`src/go2_mujoco_env.py` 의  `_get_reward` method 참고)

<br>


Reward 함수와 종료 조건은 다음 모듈에 분리되어 구현되어 있습니다.

- `src/mdp/reward.py`
- `src/mdp/termination.py`

Go2 환경에서는 Go1 대비 추가적인 reward/cost 항목이 포함됩니다.

| 항목 | 설명 |
|------|------|
| `gait_enforcement` | trot gait 패턴을 따르도록 발 접지 타이밍 유도 |
| `foot_clearance` | 발이 지면에서 적절한 높이로 들리도록 유도 |
| `hip_spread` | 고관절(hip/abad) 관절이 기본 자세에서 벗어나지 않도록 패널티 |
| `joint_pos_deviation` | 관절이 기본 자세에서 크게 벗어나는 것에 패널티 |

In [ ]:
show_func(go2_env.Go2MujocoEnv._get_reward, max_height=800)

In [ ]:
show_code(f"{repo_dir}/src/mdp/reward.py", max_height=800)

In [ ]:
# Check observation / action space
import importlib
import numpy as np
import src.go2_mujoco_env as go2_env

importlib.reload(go2_env)

# Create environment (no rendering)
env = go2_env.Go2MujocoEnv(
    prj_path=repo_dir,
    render_mode=None,
)

obs, info = env.reset()

print(f"Observation shape: {np.array(obs).shape}\n")
print(f"Action space: {env.action_space}\n")
print(f"Observation space: {env.observation_space}\n")

# Take one random step
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

print("One-step reward:", reward)
print("Terminated:", terminated)

env.close()

In [ ]:
import numpy as np
import importlib

import os
import gc
import time
import imageio
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv
from IPython.display import Video, display
from pathlib import Path

import src.go2_mujoco_env as go2_env

from src.utils.reward_logging_callback import RewardLoggingCallback

policy_cfg_path = Path(repo_dir + "/src/params.yaml")
with policy_cfg_path.open("r", encoding="utf-8") as f:
    policy_cfg = yaml.safe_load(f)

In [ ]:
# colab에서 실행하기 위한 설정
policy_cfg['n_envs'] = 12
policy_cfg['policy']['batch_size'] = 64

print(policy_cfg['n_envs'])
print(policy_cfg['policy']['batch_size'])

---

## 2. PPO Code Review

### 2.1 Stable-Baselines3

본 튜토리얼에서는 **Stable-Baselines3(SB3)** 라이브러리에 구현된
**PPO(Proximal Policy Optimization)** 알고리즘을 사용하였습니다.

SB3는 PyTorch 기반의 라이브러리로,
PPO, SAC, TD3 등의 다양한 강화학습 알고리즘을 제공합니다.

PPO는 actor-critic 구조를 기반으로 하며,
정책 업데이트 시 변화 폭을 제한(clipping)하여
학습 안정성을 확보하는 것이 특징입니다.

이 섹션에서는 수식적 유도보다는,
SB3의 PPO 모듈이 이번 보행제어기의 학습과정에서 어떻게 활용되었는지에 초점을 맞춥니다.


In [ ]:
import stable_baselines3
import stable_baselines3.ppo.ppo as ppo_module
import inspect, os

print("stable-baselines3 version:", stable_baselines3.__version__)
print("PPO module path:", os.path.abspath(ppo_module.__file__))
print("PPO class:", ppo_module.PPO)

<br>

### 2.2 Vectorized Environments

PPO는 on-policy 알고리즘이므로,
매 업데이트마다 많은 샘플을 필요로 합니다.

이를 위해 SB3는 여러 개의 환경을 동시에 실행하는
**vectorized environment**를 지원합니다.

본 튜토리얼에서는 `SubprocVecEnv`를 사용하여
여러 개의 Go2 환경을 병렬로 실행합니다.
이는 MuJoCo 기반 시뮬레이션에서 샘플 수집 속도를
크게 향상시켜 줍니다.

<br>

### 2.3 PPO 에이전트 생성

이제 병렬 환경을 기반으로 PPO 에이전트를 생성합니다.

PPO의 정책 네트워크(actor)와 가치 함수(critic)는
SB3 내부에서 자동으로 생성됩니다.

<br>

### 2.4 PPO 학습 루프

PPO 에이전트가 생성되면,
`learn()` 함수를 통해 학습을 수행합니다.

`learn()` 내부에서는 다음 과정이 반복됩니다.

1. 현재 정책으로 병렬 환경과 상호작용하며 rollout 수집
2. GAE를 이용한 advantage 계산
3. 정책 및 가치 함수 업데이트
4. 학습 로그 기록 및 콜백 실행

In [ ]:
show_func(ppo_module.PPO.train)

---
## 3. Training and Finetuning

사족 보행 로봇의 보행 정책을 학습합니다.
본 튜토리얼에서는 스크래치부터 학습을 수행하는 방법과, 사전학습된 정책을 초기값으로 사용하여 추가 학습(파인튜닝)을 수행하는 방법의 두 가지 방식을 제공합니다.

- `USE_PRETRAINED = True`이면, 사전학습된 체크포인트를 불러온 뒤 학습을 이어서 진행합니다.
- `USE_PRETRAINED = False`이면, 정책을 처음부터 초기화하여 학습합니다.

기본 설정은 스크래치부터 학습을 진행하는 방식입니다.
다만 본 실습에서는 시간적 제약으로 인해 미리 학습된 정책을 사용하며, 학습 과정은 스킵합니다.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {repo_dir}/logs

In [ ]:
importlib.reload(go2_env)

USE_PRETRAINED = False
PRETRAINED_MODEL_PATH = f"{repo_dir}/models/Go2_eighth_test/final_model.zip"

# Train
MODEL_DIR = "models"
LOG_DIR = "logs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

vec_env = make_vec_env(
    go2_env.Go2MujocoEnv,
    env_kwargs={"prj_path": repo_dir},
    n_envs=policy_cfg["n_envs"],
    seed=policy_cfg["seed"],
    vec_env_cls=SubprocVecEnv,
)

train_time = time.strftime("%Y-%m-%d_%H-%M-%S")
run_name = f"{train_time}"

model_path = f"{MODEL_DIR}/{run_name}"
print(
    f"Training on {policy_cfg['n_envs']} parallel training environments and saving models to '{model_path}'"
)

checkpoint_callback = CheckpointCallback(
    save_freq=policy_cfg["policy"]["n_steps"] * policy_cfg["log"]["interval"],
    save_path=model_path,
    name_prefix="model",
    save_replay_buffer=False,
    save_vecnormalize=False,
)

eval_callback = EvalCallback(
    vec_env,
    best_model_save_path=model_path,
    log_path=LOG_DIR,
    eval_freq=policy_cfg["eval_freq"],
    n_eval_episodes=5,
    deterministic=True,
    render=False,
)

reward_logging_callback = RewardLoggingCallback()

callbacks = CallbackList([
    eval_callback,
    checkpoint_callback,
    reward_logging_callback,
])

if USE_PRETRAINED:
    print(f"Using Pretrained model from {PRETRAINED_MODEL_PATH}")
    # numpy 버전 불일치 시 space 직렬화 실패를 우회
    _dummy_env = go2_env.Go2MujocoEnv(prj_path=repo_dir, render_mode=None)
    _custom_objects = {
        "observation_space": _dummy_env.observation_space,
        "action_space": _dummy_env.action_space,
    }
    _dummy_env.close()
    model = PPO.load(path=PRETRAINED_MODEL_PATH, env=vec_env,
                custom_objects=_custom_objects,
                learning_rate=policy_cfg["policy"]["learning_rate"],
                n_steps=policy_cfg["policy"]["n_steps"],
                batch_size=policy_cfg["policy"]["batch_size"],
                n_epochs=policy_cfg["policy"]["n_epochs"],
                gamma=policy_cfg["policy"]["gamma"],
                gae_lambda=policy_cfg["policy"]["gae_lambda"],
                clip_range=policy_cfg["policy"]["clip_range"],
                normalize_advantage=policy_cfg["policy"]["normalize_advantage"],
                ent_coef=policy_cfg["policy"]["ent_coef"],
                vf_coef=policy_cfg["policy"]["vf_coef"],
                max_grad_norm=policy_cfg["policy"]["max_grad_norm"],
                verbose=1,
                tensorboard_log=LOG_DIR)
    model.tensorboard_log = LOG_DIR
else:
    print("Training from Network model from scratch")
    model = PPO("MlpPolicy",
                env=vec_env,
                learning_rate=policy_cfg["policy"]["learning_rate"],
                n_steps=policy_cfg["policy"]["n_steps"],
                batch_size=policy_cfg["policy"]["batch_size"],
                n_epochs=policy_cfg["policy"]["n_epochs"],
                gamma=policy_cfg["policy"]["gamma"],
                gae_lambda=policy_cfg["policy"]["gae_lambda"],
                clip_range=policy_cfg["policy"]["clip_range"],
                normalize_advantage=policy_cfg["policy"]["normalize_advantage"],
                ent_coef=policy_cfg["policy"]["ent_coef"],
                vf_coef=policy_cfg["policy"]["vf_coef"],
                max_grad_norm=policy_cfg["policy"]["max_grad_norm"],
                verbose=1,
                tensorboard_log=LOG_DIR)

model.learn(
    total_timesteps=policy_cfg["total_timestep"],
    reset_num_timesteps=True,
    progress_bar=True,
    tb_log_name=run_name,
    callback=callbacks,
)
# Save final model
model.save(f"{model_path}/final_model")

vec_env.close()

del model
del eval_callback
del vec_env

gc.collect()

## 4. Policy Evaluation

학습이 완료된 후, 학습된 정책을 단일 환경에서 평가합니다.

로봇의 동작을 렌더링하여 영상으로 저장하고, 정성적(qualitative) 분석에 활용합니다.

- Control frequency: 50 Hz
- Video frame rate: 10 FPS
- 프레임을 일정 주기로 샘플링하여 MP4 파일로 저장합니다.

아래 셀은 롤아웃(rollout) 영상을 생성하고 표시합니다.

In [ ]:
model_name = "Go2_eighth_test"
# model_name = "Go2_seventh_test"
# model_name = "Go2_sixth_test"

In [ ]:
# Test
import time
from tqdm.auto import tqdm

ep_len = 0
importlib.reload(go2_env)
model_path = f"{repo_dir}/models/{model_name}/best_model.zip"
print(f"Loading model from {model_path}")
WIDTH, HEIGHT = 320, 240

# Set a fixed command for testing [vx (m/s), vy (m/s), wz (rad/s)]
given_command = [0.9, 0.0, 0.0]

env = go2_env.Go2MujocoEnv(
    prj_path=f"{repo_dir}",
    given_command=given_command,
    render_mode="rgb_array",
    camera_name="tracking",
    width=WIDTH,
    height=HEIGHT,
)

env._reset_noise_scale = 0.05  # reduce initial random noise

# numpy 버전 불일치 시 action/observation space 직렬화 실패를 우회
custom_objects = {
    "observation_space": env.observation_space,
    "action_space": env.action_space,
}

model = PPO.load(path=model_path, env=env, verbose=1, custom_objects=custom_objects)

video_path = f"{repo_dir}/../rollout_{model_name}.mp4"

obs, _ = env.reset()
max_time_step_s = policy_cfg["test"]["max_time_step_s"]
ep_len = 0
t_render = 0.0
n_render = 0
last_render = 0.0
start = time.perf_counter()
video_fps = 10

# Ctrl Hz: 50
render_interval = 50 // video_fps
max_steps = int(max_time_step_s * 50)

frames = []
pbar = tqdm(total=max_steps, desc="rollout", unit="step", dynamic_ncols=True)

print("max time:", max_time_step_s)
print("max step: ", max_steps)

while ep_len < max_steps:
    with torch.no_grad():
        action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)

    if ep_len % render_interval == 0:
        t0 = time.perf_counter()
        frame = env.render()
        frames.append(frame)
        last_render = time.perf_counter() - t0
        t_render += last_render
        n_render += 1

    # status bar 업데이트
    elapsed = time.perf_counter() - start
    steps_per_sec = ep_len / max(elapsed, 1e-9)
    avg_render = (t_render / n_render) if n_render else 0.0

    pbar.set_postfix({
        "steps/s": f"{steps_per_sec:6.1f}",
        "renders": n_render,
        "r_last(s)": f"{last_render:5.3f}",
        "r_avg(s)": f"{avg_render:5.3f}",
    })
    pbar.update(1)

    if terminated or truncated:
        print(f"episode finished: ep_len={ep_len}, reward={reward:.3f}")
        obs, _ = env.reset()

    ep_len += 1

imageio.mimwrite(
    video_path,
    frames,
    fps=video_fps,
    codec="libx264",
    quality=8,
    pixelformat="yuv420p",
)

env.close()

print("avg render sec:", t_render / max(n_render, 1))
print("Saved video to:", video_path)

In [ ]:
from IPython.display import Video, display

print(f"model: {model_name}")
display(
    Video(
        video_path,
        embed=True,
        html_attributes="controls autoplay loop"
    )
)

## 5. 정리

이 노트북에서는 Unitree Go2 로봇의 보행 제어기를 PPO 알고리즘으로 학습하는 전체 파이프라인을 살펴보았습니다.

Go1 대비 Go2 환경의 주요 차이점:

- **Gait phase tracking**: trot gait 패턴을 위한 위상 인코딩이 observation에 포함됨
- **추가 reward 항목**: `gait_enforcement`, `foot_clearance`, `hip_spread`, `joint_pos_deviation`
- **Action space**: 절대 위치가 아닌 기본 자세 대비 **상대적 오프셋**으로 정의됨
- **Early termination penalty**: 로봇이 넘어질 경우 -100 보상
